# k-nearest neighbors

_Machine Learning_

**To predict a new point, look at the closest known points.**

k-nearest neighbors (k-NN) makes no model at all during training. It just stores the data.

     To predict a new point, find the $k$ closest stored points.

---

This notebook builds k-NN up one step at a time: first see how the choice of `k` changes accuracy on synthetic data, then sweep `k` on real tumour scans to find the sweet spot. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We try k-NN on a synthetic dataset in two steps: (1) split the data into train and test sets, then (2) fit a scaled k-NN for several values of `k` and compare test accuracy.

### Step 1 — Make data and split into train / test

`make_classification` builds 400 labelled examples with 4 features. We hold out 25% as a **test set** so we can measure accuracy on points the model never saw — the only honest way to judge a method that simply memorises its training data.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# 400 labelled examples with 4 features.
X, y = make_classification(n_samples=400, n_features=4, random_state=0)

# Hold out 25% to test on unseen points.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit scaled k-NN for several k

k-NN decides by distance, so we **standardise** features first (inside a pipeline) to stop large-scale columns from dominating. We sweep `k`: a small `k` follows noise (overfits), a large `k` averages over too many neighbours (oversmooths).

In [ ]:
# Scaling + kNN, since kNN relies on distances.
for k in [1, 5, 15]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(Xtr, ytr)
    print("k=%2d  test accuracy: %.3f" % (k, knn.score(Xte, yte)))

## Visualize the data & results

_On breast-cancer data, how many neighbors k gives the best accuracy - too few overfits, too many oversmooths?_

### Step 1 — Load tumour scans and split

We load the real **breast-cancer** dataset and again carve off a 25% test set. We'll measure how test accuracy varies with `k` to find the value that generalises best on real measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(bc.data, bc.target, test_size=0.25, random_state=0)

### Step 2 — Sweep k and plot accuracy

For each candidate `k` we fit a scaled k-NN and record its test accuracy. Plotting accuracy against `k` reveals the trade-off: too few neighbours overfits, too many oversmooths, and the peak in between is the best choice here.

In [ ]:
ks = [1, 3, 5, 9, 15, 25]
acc = []
for k in ks:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(Xtr, ytr)
    acc.append(knn.score(Xte, yte))

plt.plot(ks, acc, color="#4ea1ff", marker="o", label="test accuracy")
plt.xlabel("k (neighbors)")
plt.ylabel("test accuracy")
plt.title("kNN accuracy vs k")
plt.legend()
plt.show()